In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"          # single GPU — avoids DataParallel OOM
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

!pip install -q -U "transformers>=4.50" "trl>=0.12" "peft>=0.13" "datasets>=2.20" \
    "bitsandbytes>=0.43" accelerate sentencepiece rouge_score sacrebleu evaluate "torchao>=0.16.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 82.5 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 863.2/863.2 kB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 27.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 55.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 85.9 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 31.6 MB/s eta 0:00:00


In [2]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
try:
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    HF_TOKEN = "hf_xxx"
login(HF_TOKEN)

In [3]:
import torch, gc
try: del trainer, model, base
except NameError: pass
gc.collect(); torch.cuda.empty_cache()

## Config

In [4]:

DATA_CSV = "/kaggle/input/datasets/prettyntakirutimana/wambaza-training-data/wambaza_training_data.csv"

BASE_MODEL   = "google/gemma-3-1b-it"
OUTPUT_DIR   = "/kaggle/working/gemma3-1b-en-kin-noswahili"
MAX_SEQ_LEN  = 384
SEED         = 42

EXCLUDE_LANGS = ["Swa"]

USE_BACKTRANSLATION_AUGMENTATION = False
BACKTRANSLATION_PIVOT = "fra_Latn"   
BACKTRANSLATION_TARGET_MULTIPLIER = 2   

In [5]:
import pandas as pd, re, numpy as np

data = pd.read_csv(DATA_CSV)
data.columns = [c.strip().lower() for c in data.columns]

print("Shape:", data.shape)
print("Columns:", list(data.columns))
print("\nBefore excluding languages:")
print(data["lang"].value_counts())

data = data[~data["lang"].isin(EXCLUDE_LANGS)].reset_index(drop=True)
print(f"\nAfter excluding {EXCLUDE_LANGS}:")
print(data["lang"].value_counts())

def clean(t):
    t = str(t)
    t = re.sub(r"as of my last (knowledge update|update)", "", t, flags=re.IGNORECASE)
    return re.sub(r"\s{2,}", " ", t).strip()
data["question"] = data["question"].map(clean)
data["answer"]   = data["answer"].map(clean)
data = data[(data["question"] != "") & (data["answer"] != "")]
data = data.drop_duplicates(subset=["lang","question","answer"]).reset_index(drop=True)
print(f"\nAfter cleaning/dedup:")
print(data["lang"].value_counts())

Shape: (19249, 4)
Columns: ['question', 'answer', 'lang', 'source']

Before excluding languages:
lang
en     7833
kin    7582
lug    3834
Name: count, dtype: int64

After excluding ['Swa']:
lang
en     7833
kin    7582
lug    3834
Name: count, dtype: int64

After cleaning/dedup:
lang
en     7831
kin    7581
lug    3834
Name: count, dtype: int64


In [6]:
eval_parts, pool_parts = [], []
for lang in data["lang"].unique():
    sub = data[data["lang"] == lang]
    eval_sub = sub.sample(frac=0.12, random_state=SEED)
    pool_sub = sub.drop(eval_sub.index)
    eval_parts.append(eval_sub)
    pool_parts.append(pool_sub)

eval_set = pd.concat(eval_parts, ignore_index=True).reset_index(drop=True)
pool = pd.concat(pool_parts, ignore_index=True).reset_index(drop=True)
print(f"Eval (held out): {len(eval_set)} | Training pool: {len(pool)}")
print(pool["lang"].value_counts())

Eval (held out): 2310 | Training pool: 16936
lang
en     6891
kin    6671
lug    3374
Name: count, dtype: int64


In [7]:
if USE_BACKTRANSLATION_AUGMENTATION:
    from transformers import AutoModelForSeq2SeqLM, AutoTokenizer as NLLBTokenizer

    minority_lang = pool["lang"].value_counts().idxmin()
    print(f"Upsampling minority language via back-translation: {minority_lang}")

    LANG_TO_NLLB = {"Eng": "eng_Latn", "Kin": "kin_Latn", "Swa": "swh_Latn", "Lug": "lug_Latn"}
    src_code = LANG_TO_NLLB[minority_lang]

    nllb_tok = NLLBTokenizer.from_pretrained("facebook/nllb-200-distilled-600M")
    nllb_model = AutoModelForSeq2SeqLM.from_pretrained(
        "facebook/nllb-200-distilled-600M", torch_dtype=torch.float16).to("cuda")
    nllb_model.eval()

    @torch.no_grad()
    def nllb_translate(texts, src_lang, tgt_lang, batch_size=16):
        nllb_tok.src_lang = src_lang
        bos = nllb_tok.convert_tokens_to_ids(tgt_lang)
        out = []
        for i in range(0, len(texts), batch_size):
            batch = [t if t.strip() else " " for t in texts[i:i+batch_size]]
            enc = nllb_tok(batch, return_tensors="pt", padding=True, truncation=True,
                           max_length=256).to("cuda")
            gen = nllb_model.generate(**enc, forced_bos_token_id=bos, max_length=256, num_beams=4)
            out.extend(nllb_tok.batch_decode(gen, skip_special_tokens=True))
        return out

    minority_rows = pool[pool["lang"] == minority_lang]
    augmented_frames = []
    for _ in range(BACKTRANSLATION_TARGET_MULTIPLIER - 1):
        pivot_q = nllb_translate(minority_rows["question"].tolist(), src_code, BACKTRANSLATION_PIVOT)
        pivot_a = nllb_translate(minority_rows["answer"].tolist(), src_code, BACKTRANSLATION_PIVOT)
        back_q = nllb_translate(pivot_q, BACKTRANSLATION_PIVOT, src_code)
        back_a = nllb_translate(pivot_a, BACKTRANSLATION_PIVOT, src_code)
        aug = pd.DataFrame({"lang": minority_lang, "question": back_q, "answer": back_a})
        augmented_frames.append(aug)

    backtranslated = pd.concat(augmented_frames, ignore_index=True)
    print(f"Generated {len(backtranslated)} back-translated variants of {minority_lang}")
    print("\nSample (original -> back-translated):")
    print("ORIG:", minority_rows["answer"].iloc[0][:150])
    print("BT  :", backtranslated["answer"].iloc[0][:150])

    pool = pd.concat([pool, backtranslated], ignore_index=True)

    del nllb_model, nllb_tok
    gc.collect(); torch.cuda.empty_cache()
    print("\nNLLB cleared from memory.")
else:
    print("Back-translation augmentation OFF — using standard duplication-based oversampling only.")

Back-translation augmentation OFF — using standard duplication-based oversampling only.


In [8]:
counts = pool["lang"].value_counts()
target = int(counts.median() * 1.2) if len(counts) > 1 else counts.max()
MAX_OVERSAMPLE = 4
print(f"Auto-computed per-language target: {target} rows")

balanced_parts = []
for lang, cnt in counts.items():
    sub = pool[pool["lang"] == lang]
    if cnt > target:
        sub = sub.sample(target, random_state=SEED)
    elif cnt < target:
        factor = min(MAX_OVERSAMPLE, max(1, target // cnt))
        sub = pd.concat([sub] * factor, ignore_index=True)
        if len(sub) > target:
            sub = sub.sample(target, random_state=SEED)
    balanced_parts.append(sub)

train_pool = pd.concat(balanced_parts, ignore_index=True).sample(frac=1, random_state=SEED).reset_index(drop=True)
print("\nFinal balanced training set:")
print(train_pool["lang"].value_counts())
print(f"Total: {len(train_pool)}")

Auto-computed per-language target: 8005 rows

Final balanced training set:
lang
en     6891
lug    6748
kin    6671
Name: count, dtype: int64
Total: 20310


In [9]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

def to_messages(row):
    return [{"role":"user","content":row["question"]}, {"role":"assistant","content":row["answer"]}]

train_pool["messages"] = train_pool.apply(to_messages, axis=1)
eval_set["messages"]   = eval_set.apply(to_messages, axis=1)

def n_tokens(msgs):
    text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
    return len(tokenizer(text).input_ids)

lens = train_pool["messages"].map(n_tokens)
print(f"Token lengths: median={int(lens.median())}, 95th={int(lens.quantile(.95))}, max={int(lens.max())}")

from datasets import Dataset
train_ds = Dataset.from_pandas(train_pool[["messages"]], preserve_index=False)
eval_ds  = Dataset.from_pandas(eval_set[["messages"]],   preserve_index=False)
print(f"\n{len(train_ds)} train / {len(eval_ds)} held-out eval")

config.json:   0%|          | 0.00/899 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

Token lengths: median=173, 95th=429, max=738

20310 train / 2310 held-out eval


In [10]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training

bnb = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True,
)
base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, quantization_config=bnb,
    torch_dtype=torch.bfloat16, attn_implementation="eager", device_map={"": 0},
)

if hasattr(base, "peft_config"):
    raise RuntimeError("base already has a PEFT adapter attached — re-run the GPU-clear "
                       "cell above, then re-run THIS cell fresh before continuing.")

base = prepare_model_for_kbit_training(base)
base.config.use_cache = False
base.enable_input_require_grads()

lora = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
)

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).
[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.00G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

In [ ]:
from trl import SFTTrainer, SFTConfig

cfg = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,        # effective batch 16
    learning_rate=2e-4,
    max_grad_norm=0.3,
    lr_scheduler_type="cosine",
    warmup_steps=20,
    logging_steps=10,
    eval_strategy="steps", eval_steps=150,
    save_strategy="steps", save_steps=150, save_total_limit=3,
    fp16=False, bf16=True,
    max_length=MAX_SEQ_LEN,
    packing=True,             # Gemma's chat template has generation markers -> assistant_only_loss works natively
    assistant_only_loss=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="paged_adamw_8bit",
    report_to="none", seed=SEED,
)
trainer = SFTTrainer(model=base, args=cfg, peft_config=lora,
                     train_dataset=train_ds, eval_dataset=eval_ds,
                     processing_class=tokenizer)
print("grad ckpt active:", trainer.model.is_gradient_checkpointing)

import time
t0 = time.time()
trainer.train()
print(f"\nTraining wall time: {(time.time()-t0)/3600:.2f} hours")

trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("Saved ->", OUTPUT_DIR)

[RANK 0] Padding-free training is enabled, but the attention implementation is not set to a supported Flash Attention variant. Padding-free training flattens batches into a single sequence, and only the following implementations are known to reliably support this: flash_attention_2, flash_attention_3, kernels-community/flash-attn2, kernels-community/flash-attn3, kernels-community/vllm-flash-attn3. Using other implementations may lead to unexpected behavior. To ensure compatibility, set `attn_implementation` in the model configuration to one of these supported options or verify that your attention mechanism can handle flattened sequences.
[RANK 0] You are using packing, but the attention implementation is not set to a supported Flash Attention variant. Packing gathers multiple samples into a single sequence, and only the following implementations are known to reliably support this: flash_attention_2, flash_attention_3, kernels-community/flash-attn2, kernels-community/flash-attn3, kernel

Tokenizing train dataset:   0%|          | 0/20310 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/20310 [00:00<?, ? examples/s]

Packing train dataset:   0%|          | 0/20310 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/2310 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/2310 [00:00<?, ? examples/s]

Packing eval dataset:   0%|          | 0/2310 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1}.


grad ckpt active: True


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
150,2.004482,1.936395,1.892633,913057.000000,0.576183
300,1.637490,1.502567,1.537927,1825681.000000,0.658321
450,1.236847,1.246131,1.281845,2739605.000000,0.710631
600,1.082291,1.065221,1.082690,3651842.000000,0.749612
750,0.860099,0.935963,0.861342,4562902.000000,0.782763
900,0.757964,0.835111,0.769179,5476551.000000,0.807366
1050,0.573537,0.736429,0.690286,6389437.000000,0.830771
1200,0.516984,0.671008,0.663679,7301957.000000,0.846029
1350,0.396944,0.635017,0.560951,8213364.000000,0.857877
1500,0.330195,0.615804,0.526441,9126224.000000,0.864101
